## 面试问题

循环内 token 预算怎么记账与逼近上限时降级？

## 回答主线

每步消耗 token，必须在状态里显式记账剩余预算，逼近上限时主动降级（压缩历史→缩短输出→提前收敛），而非硬撞上限崩溃。本 Notebook 用预算 100、消耗随历史增长的模型，对比不记账（历史累积到某步超预算失败）与记账+降级（低于水位线时压缩历史，安全完成）。

## 真实案例

预算 100，每步消耗 = 基础 8 + 历史长度 x 5。不记账时历史无限增长，消耗迅速超预算；记账时剩余低于水位线就压缩历史，把每步消耗压回低位。数据为教学消耗模型，不代表真实 token 计费。

In [1]:
TOTAL_BUDGET = 100  # 循环的总 token 预算。
BASE_COST = 8  # 每步基础消耗。
GROWTH = 5  # 每积累一步历史增加的消耗。
NUM_STEPS = 6  # 计划执行的步数。

print("总预算:", TOTAL_BUDGET)  # 展示总预算。
print("每步消耗 = 基础", BASE_COST, "+ 历史长度 x", GROWTH)  # 展示消耗模型。
print("计划步数:", NUM_STEPS)  # 展示计划步数。

总预算: 100
每步消耗 = 基础 8 + 历史长度 x 5
计划步数: 6


## 基线（Baseline）

反面基线：不记账。历史无限增长，每步消耗越来越大，累积到某步就超出总预算，被迫失败。

In [2]:
def run_no_accounting(num_steps):  # 不记账：历史无限增长，消耗累积直到超预算。
    history_len = 0  # 历史长度。
    spent = 0  # 已消耗总量。
    for step in range(1, num_steps + 1):  # 逐步执行。
        cost = BASE_COST + history_len * GROWTH  # 本步消耗随历史增长。
        spent += cost  # 累加消耗。
        history_len += 1  # 历史增长一步。
        if spent > TOTAL_BUDGET:  # 超出预算。
            return {"status": "budget_exceeded", "step": step, "spent": spent}  # 撞上限失败。
    return {"status": "completed", "spent": spent}  # 正常完成。

no_acc = run_no_accounting(NUM_STEPS)  # 运行不记账循环。
print("不记账结果:", no_acc)  # 展示第几步撞上预算上限。

不记账结果: {'status': 'budget_exceeded', 'step': 6, 'spent': 123}


## 失败案例与修正

不记账在第 6 步撞上预算。修正是记账+降级：每步先算剩余预算，低于水位线(45)就把历史压缩为一条摘要，历史长度回落、每步消耗随之下降，从而安全完成全部步骤。

In [3]:
def run_with_degradation(num_steps, low_water=45):  # 记账+降级：预算不足时压缩历史降低消耗。
    history_len = 0  # 历史长度。
    spent = 0  # 已消耗总量。
    compressions = 0  # 记录降级次数。
    for step in range(1, num_steps + 1):  # 逐步执行。
        remaining = TOTAL_BUDGET - spent  # 计算剩余预算。
        if remaining < low_water:  # 剩余预算低于水位线触发降级。
            history_len = 1  # 压缩历史为一条摘要。
            compressions += 1  # 累加降级次数。
        cost = BASE_COST + history_len * GROWTH  # 本步消耗随压缩后的历史。
        spent += cost  # 累加消耗。
        history_len += 1  # 历史增长一步。
    return {"status": "completed", "spent": spent, "compressions": compressions}  # 返回完成结果与降级次数。

with_deg = run_with_degradation(NUM_STEPS)  # 运行记账降级循环。
print("记账降级结果:", with_deg)  # 展示通过压缩历史安全完成全部步数。

记账降级结果: {'status': 'completed', 'spent': 88, 'compressions': 2}


In [4]:
print("不记账:", no_acc["status"], "在第", no_acc.get("step"), "步撞上预算")  # 不记账撞上限。
print("记账降级:", with_deg["status"], "共降级", with_deg["compressions"], "次")  # 降级完成。
print("最终消耗 不记账 vs 降级:", no_acc["spent"], with_deg["spent"])  # 对比总消耗。

不记账: budget_exceeded 在第 6 步撞上预算
记账降级: completed 共降级 2 次
最终消耗 不记账 vs 降级: 123 88


## 结果解读

不记账在第 6 步消耗累积到超预算而失败；记账降级在剩余低于水位线时压缩历史，每步消耗回落，最终总消耗控制在预算内并完成全部步骤。要点：预算入状态、降级分级且可观测，阈值需用回归集调（题 30）。

In [5]:
deg_util = with_deg["spent"] / TOTAL_BUDGET  # 降级方案的预算占用比。
no_acc_util = no_acc["spent"] / TOTAL_BUDGET  # 不记账的预算占用比。
print("不记账预算占用:", round(no_acc_util, 2), "(超过 1 表示超支)")  # 展示不记账超支。
print("降级方案预算占用:", round(deg_util, 2), "(<=1 表示在预算内)")  # 展示降级在预算内。
print("降级方案是否在预算内:", with_deg["spent"] <= TOTAL_BUDGET)  # 展示降级安全。

不记账预算占用: 1.23 (超过 1 表示超支)
降级方案预算占用: 0.88 (<=1 表示在预算内)
降级方案是否在预算内: True


In [6]:
assert no_acc["status"] == "budget_exceeded"  # 不记账必然撞上预算上限。
assert with_deg["status"] == "completed"  # 记账降级安全完成全部步数。
assert with_deg["compressions"] >= 1  # 降级过程触发了至少一次历史压缩。
assert with_deg["spent"] <= TOTAL_BUDGET  # 降级方案的总消耗不超预算。
assert no_acc["spent"] > TOTAL_BUDGET  # 不记账的总消耗超出预算。
print("全部不变量通过")  # 输出测试通过信号。

全部不变量通过
